# Capstone Conference Paper — COMPLETE FIXED COLAB

**Project:** Agentic Explainable Agricultural Decision Support System using Multi-Agent AI, RAG and Machine Learning

This version is fully self-contained. All code is visible in this notebook. It fixes the NumPy `_slice` import problem by validating/repairing the scientific stack **before NumPy is imported**, then installs NLP packages without allowing pip to replace NumPy again.

### Run instructions
1. Open this notebook in a **fresh** Colab tab.
2. `Runtime → Change runtime type → T4 GPU`.
3. `Runtime → Run all`.
4. Allow Google Drive access when prompted.

Everything is saved automatically to `MyDrive/Capstone_Conference_Paper/`.


In [ ]:
# ============================================================
# 0. NUMPY-SAFE DEPENDENCY SETUP
# ============================================================
import sys, subprocess

# Check the scientific stack in a separate Python process so this notebook
# does not import a broken NumPy into the live kernel.
check = subprocess.run(
    [sys.executable, '-c', 'import numpy,pandas,scipy,sklearn; print(numpy.__version__, pandas.__version__, scipy.__version__, sklearn.__version__)'],
    capture_output=True, text=True
)

if check.returncode != 0:
    print('Scientific stack is broken; repairing compatible wheels...')
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', '--force-reinstall',
        'numpy==2.2.6', 'scipy==1.15.3', 'pandas==2.2.3', 'scikit-learn==1.6.1'
    ], check=True)
else:
    print('Scientific stack OK:', check.stdout.strip())

# Install the NLP stack WITHOUT dependency resolution, so pip cannot swap
# NumPy/SciPy/Pandas/Scikit-learn underneath the running Colab kernel.
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--no-cache-dir',
    'transformers==4.51.3',
    'sentence-transformers==4.1.0',
    'accelerate==1.6.0',
    'huggingface_hub==0.30.2',
    'tokenizers==0.21.1',
    'safetensors==0.5.3',
    'openpyxl==3.1.5',
    'xlrd==2.0.2'
], check=True)

print('Dependency setup complete.')


In [ ]:
# ============================================================
# 1. COMPLETE CONFERENCE-PAPER PIPELINE
# ============================================================
import os, re, json, time, random, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from tqdm.auto import tqdm
from huggingface_hub import snapshot_download
from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import ndcg_score
from google.colab import drive

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DATASET_REPO = 'CGIAR/RAG-Chunk-Analysis'
MODEL_NAME = 'Qwen/Qwen3-4B-Instruct-2507'
BI_ENCODER_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
CROSS_ENCODER_NAME = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
TOP_K = 5
MIN_COVERAGE = 0.90
MAX_GENERATION_QUERIES = None   # None = full untouched test set
MAX_CONTEXT_CHARS = 1200
MAX_INPUT_TOKENS = 4096
MAX_NEW_TOKENS = 96
DRIVE_DIR = '/content/drive/MyDrive/Capstone_Conference_Paper'

print('NumPy:', np.__version__, '| Pandas:', pd.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Please select Runtime > Change runtime type > T4 GPU, then Run all again.')
print('GPU:', torch.cuda.get_device_name(0))

drive.mount('/content/drive', force_remount=False)
SAVE_DIR = Path(DRIVE_DIR); SAVE_DIR.mkdir(parents=True, exist_ok=True)
print('Saving all results to:', SAVE_DIR)

# ---------------- DATA DOWNLOAD ----------------
DATA_DIR = Path('/content/CGIAR_RAG_Chunk_Analysis')
snapshot_download(repo_id=DATASET_REPO, repo_type='dataset', local_dir=str(DATA_DIR))
excel_files = sorted(list(DATA_DIR.rglob('*.xlsx')) + list(DATA_DIR.rglob('*.xls')))
frames = []
for f in excel_files:
    try:
        for sheet, df in pd.read_excel(f, sheet_name=None).items():
            if df is None or df.empty: continue
            t = df.copy(); t['source_file'] = f.name; t['source_sheet'] = str(sheet); frames.append(t)
    except Exception as e:
        print('Skipped:', f.name, '|', e)
if not frames:
    raise RuntimeError('No readable dataset sheets found.')
raw_df = pd.concat(frames, ignore_index=True, sort=False)

def clean_text(x):
    if pd.isna(x): return None
    s = str(x).strip()
    return None if (not s or s.lower() == 'nan') else s

def rel_label(x):
    if pd.isna(x): return np.nan
    s = str(x).strip().lower()
    if 'irrelevant' in s or 'not relevant' in s: return 0
    if 'relevant' in s: return 1
    if s in {'yes','y','1','true'}: return 1
    if s in {'no','n','0','false'}: return 0
    try:
        v = float(x); return int(v) if v in (0,1) else np.nan
    except Exception:
        return np.nan

chunk_cols = []
for c in raw_df.columns:
    m = re.fullmatch(r'Chunk\s+(\d+)', str(c).strip())
    if m: chunk_cols.append((int(m.group(1)), c))
chunk_cols = sorted(chunk_cols)
if not chunk_cols: raise ValueError('No Chunk N columns detected.')

rows = []
for rid, row in raw_df.iterrows():
    q = clean_text(row.get('User Query')); resp = clean_text(row.get('Response'))
    if q is None: continue
    for n, ccol in chunk_cols:
        suffix = '' if n == 1 else f'.{n-1}'
        chunk = clean_text(row.get(ccol))
        if chunk is None: continue
        rcol = f'Relevant/ Irrelevant{suffix}'
        rows.append({
            'original_row_id': rid, 'query': q, 'original_response': resp,
            'chunk_number': n, 'chunk': chunk,
            'source': clean_text(row.get(f'Source{suffix}')),
            'page': row.get(f'Page{suffix}') if f'Page{suffix}' in raw_df.columns else np.nan,
            'relevance_raw': row.get(rcol) if rcol in raw_df.columns else np.nan,
            'relevance_label': rel_label(row.get(rcol) if rcol in raw_df.columns else np.nan),
            'relevant_portion': clean_text(row.get(f'Relevant_portion{suffix}')) if f'Relevant_portion{suffix}' in raw_df.columns else None,
            'source_file': row.get('source_file'), 'source_sheet': row.get('source_sheet')
        })

rag_df = pd.DataFrame(rows).drop_duplicates(subset=['query','chunk','chunk_number']).reset_index(drop=True)
rag_df.to_csv(SAVE_DIR/'CGIAR_RAG_Long_Format.csv', index=False)
eval_df = rag_df.dropna(subset=['query','chunk','relevance_label']).copy()
eval_df = eval_df[eval_df['relevance_label'].isin([0,1])].drop_duplicates(subset=['query','chunk'])
valid = eval_df.groupby('query')['relevance_label'].sum()
eval_df = eval_df[eval_df['query'].isin(valid[valid > 0].index)].copy()
print('Judged query-chunk pairs:', len(eval_df), '| Queries:', eval_df['query'].nunique())

# ---------------- R0 / R1 / R2 ----------------
bi_encoder = SentenceTransformer(BI_ENCODER_NAME)
uqs = eval_df['query'].unique().tolist(); ucs = eval_df['chunk'].unique().tolist()
qe = bi_encoder.encode(uqs, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
ce = bi_encoder.encode(ucs, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
qmap = dict(zip(uqs, qe)); cmap = dict(zip(ucs, ce))
eval_df['embedding_similarity'] = [float(np.dot(qmap[q], cmap[c])) for q,c in zip(eval_df['query'], eval_df['chunk'])]
cross_encoder = CrossEncoder(CROSS_ENCODER_NAME)
eval_df['cross_encoder_score'] = cross_encoder.predict(list(zip(eval_df['query'],eval_df['chunk'])), batch_size=32, show_progress_bar=True)

def rank_metrics(g, col, ascending):
    ranked = g.sort_values(col, ascending=ascending)
    labels = ranked['relevance_label'].astype(int).tolist(); total = sum(labels); k = min(TOP_K, len(labels)); top = labels[:k]
    rr = 0.0
    for i,l in enumerate(labels,1):
        if l == 1: rr = 1/i; break
    nd = ndcg_score(np.array([labels]), np.array([list(range(len(labels),0,-1))]), k=k) if len(labels)>1 else float(labels[0])
    return {'Hit@1': labels[0], 'MRR': rr, f'Precision@{TOP_K}': sum(top)/k, f'Recall@{TOP_K}': sum(top)/total, f'NDCG@{TOP_K}': nd}

def eval_rank(df, col, ascending):
    out = []
    for q,g in df.groupby('query'):
        m = rank_metrics(g,col,ascending); m['query'] = q; out.append(m)
    return pd.DataFrame(out)

r0 = eval_rank(eval_df,'chunk_number',True); r1 = eval_rank(eval_df,'embedding_similarity',False); r2 = eval_rank(eval_df,'cross_encoder_score',False)
mc = ['Hit@1','MRR',f'Precision@{TOP_K}',f'Recall@{TOP_K}',f'NDCG@{TOP_K}']
retrieval_summary = pd.DataFrame({
    'R0 Original Ranking': r0[mc].mean(),
    'R1 MiniLM Bi-Encoder': r1[mc].mean(),
    'R2 Cross-Encoder': r2[mc].mean()
})
display(retrieval_summary.round(4))
eval_df.to_csv(SAVE_DIR/'R2_all_scored_query_chunk_pairs.csv',index=False)
r0.to_csv(SAVE_DIR/'R0_original_ranking_metrics.csv',index=False); r1.to_csv(SAVE_DIR/'R1_minilm_reranking_metrics.csv',index=False); r2.to_csv(SAVE_DIR/'R2_cross_encoder_metrics.csv',index=False)
retrieval_summary.to_csv(SAVE_DIR/'R0_R1_R2_comparison.csv')

# ---------------- CONFIDENCE ESTIMATOR ----------------
gate_df = eval_df.copy(); gate_df['original_rank_signal'] = 1.0/gate_df['chunk_number']
gate_df['bi_rank'] = gate_df.groupby('query')['embedding_similarity'].rank(method='first',ascending=False)
gate_df['cross_rank'] = gate_df.groupby('query')['cross_encoder_score'].rank(method='first',ascending=False)
gate_df['bi_rank_signal'] = 1.0/gate_df['bi_rank']; gate_df['cross_rank_signal'] = 1.0/gate_df['cross_rank']
FEATURES = ['original_rank_signal','embedding_similarity','cross_encoder_score','bi_rank_signal','cross_rank_signal']; TARGET = 'relevance_label'
s1 = GroupShuffleSplit(n_splits=1,test_size=.30,random_state=SEED); tr_idx,tmp_idx = next(s1.split(gate_df,groups=gate_df['query']))
train_df = gate_df.iloc[tr_idx].copy(); temp_df = gate_df.iloc[tmp_idx].copy()
s2 = GroupShuffleSplit(n_splits=1,test_size=.50,random_state=SEED); v_idx,t_idx = next(s2.split(temp_df,groups=temp_df['query']))
val_df = temp_df.iloc[v_idx].copy(); test_df = temp_df.iloc[t_idx].copy()
a=set(train_df['query']); b=set(val_df['query']); c=set(test_df['query']); assert not(a&b or a&c or b&c), 'Query leakage detected.'
gate_model = Pipeline([('scaler',StandardScaler()),('classifier',LogisticRegression(class_weight='balanced',max_iter=2000,random_state=SEED))])
gate_model.fit(train_df[FEATURES], train_df[TARGET].astype(int))
val_prob = gate_model.predict_proba(val_df[FEATURES])[:,1]; test_prob = gate_model.predict_proba(test_df[FEATURES])[:,1]
print('Train / Validation / Test queries:', train_df['query'].nunique(), val_df['query'].nunique(), test_df['query'].nunique())

# ---------------- VALIDATION THRESHOLD ----------------
val_work = val_df.copy(); val_work['evidence_confidence'] = val_prob
curve = []
for th in np.arange(.20,.91,.05):
    qrows = []
    for q,g in val_work.groupby('query'):
        acc = g[g['evidence_confidence'] >= th]; covered = int(len(acc)>0); total = g['relevance_label'].sum()
        qrows.append({
            'covered':covered,
            'precision':acc['relevance_label'].mean() if covered else np.nan,
            'recall':acc['relevance_label'].sum()/total if total>0 else np.nan,
            'top1':int(acc.sort_values('evidence_confidence',ascending=False).iloc[0]['relevance_label']) if covered else np.nan
        })
    z = pd.DataFrame(qrows); cov = z['covered'].mean()
    curve.append({'threshold':round(float(th),2),'coverage':cov,'abstention_rate':1-cov,'selective_precision':z['precision'].mean(skipna=True),'selective_recall':z['recall'].mean(skipna=True),'top1_relevance':z['top1'].mean(skipna=True)})
coverage_curve = pd.DataFrame(curve)
eligible = coverage_curve[coverage_curve['coverage'] >= MIN_COVERAGE]
if eligible.empty: raise RuntimeError('No threshold satisfies the coverage constraint.')
best = eligible.sort_values(['selective_precision','top1_relevance'],ascending=False).iloc[0]
FINAL_GATE_THRESHOLD = float(best['threshold'])
print('FINAL GATE THRESHOLD:', FINAL_GATE_THRESHOLD)

# ---------------- TEST GATE ----------------
test_df = test_df.copy(); test_df['evidence_confidence'] = test_prob; test_df['gate_decision'] = np.where(test_df['evidence_confidence']>=FINAL_GATE_THRESHOLD,'ACCEPT','REJECT')
qrows = []
for q,g in test_df.groupby('query'):
    total = int(g['relevance_label'].sum()); base = g.sort_values('chunk_number').head(TOP_K); acc = g[g['evidence_confidence']>=FINAL_GATE_THRESHOLD].sort_values('evidence_confidence',ascending=False).head(TOP_K); covered = int(len(acc)>0)
    qrows.append({
        'query':q,
        'baseline_precision':base['relevance_label'].mean(),
        'baseline_recall':base['relevance_label'].sum()/total,
        'baseline_top1':int(base.iloc[0]['relevance_label']),
        'gate_precision':acc['relevance_label'].mean() if covered else np.nan,
        'gate_recall':acc['relevance_label'].sum()/total if covered else 0.0,
        'gate_top1':int(acc.iloc[0]['relevance_label']) if covered else np.nan,
        'covered':covered,'abstained':1-covered,'accepted_chunks':len(acc)
    })
gate_query = pd.DataFrame(qrows)
gate_summary = pd.DataFrame({
    'Metric':['Evidence Precision','Evidence Recall','Top-1 Relevance','Query Coverage','Abstention Rate'],
    'Original Retrieval':[gate_query['baseline_precision'].mean(),gate_query['baseline_recall'].mean(),gate_query['baseline_top1'].mean(),1.0,0.0],
    'Proposed Confidence Gate':[gate_query['gate_precision'].mean(skipna=True),gate_query['gate_recall'].mean(),gate_query['gate_top1'].mean(skipna=True),gate_query['covered'].mean(),gate_query['abstained'].mean()]
})
display(gate_summary.round(4))
coverage_curve.to_csv(SAVE_DIR/'P1_validation_confidence_coverage_curve.csv',index=False); test_df.to_csv(SAVE_DIR/'P1_chunk_level_confidence_results.csv',index=False); gate_query.to_csv(SAVE_DIR/'P1_query_level_gate_results.csv',index=False); gate_summary.to_csv(SAVE_DIR/'P1_gate_vs_original_summary.csv',index=False)

# Save figures
ax = retrieval_summary.plot(kind='bar',figsize=(10,5)); ax.set_ylim(0,1); ax.set_title('R0 vs R1 vs R2 Retrieval Comparison'); plt.tight_layout(); plt.savefig(SAVE_DIR/'Figure_R0_R1_R2_Retrieval_Comparison.png',dpi=220); plt.show()
fig,ax = plt.subplots(figsize=(8,5))
for col,label in [('selective_precision','Precision'),('selective_recall','Recall'),('coverage','Coverage')]: ax.plot(coverage_curve['threshold'],coverage_curve[col],marker='o',label=label)
ax.axvline(FINAL_GATE_THRESHOLD,linestyle='--',label=f'Selected={FINAL_GATE_THRESHOLD:.2f}'); ax.set_ylim(0,1.02); ax.set_title('Validation Confidence-Coverage Trade-off'); ax.legend(); plt.tight_layout(); plt.savefig(SAVE_DIR/'Figure_Confidence_Coverage_Tradeoff.png',dpi=220); plt.show()

# Free cross-encoder GPU memory before Qwen. Keep MiniLM on CPU for final proxy evaluation.
del cross_encoder
try: bi_encoder.to('cpu')
except Exception: pass
torch.cuda.empty_cache()

# ---------------- QWEN END-TO-END ----------------
from transformers import AutoTokenizer, AutoModelForCausalLM
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
llm = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map='auto')
llm.eval()
print('Qwen device:', next(llm.parameters()).device)
if 'cuda' not in str(next(llm.parameters()).device): raise RuntimeError('Qwen did not load on the GPU.')
SYSTEM_PROMPT = '''You are an agricultural advisory assistant. Answer the farmer question using ONLY the supplied evidence. Do not invent facts or use unsupported outside knowledge. If evidence is insufficient, say: Insufficient evidence to provide a reliable answer. Keep the answer concise and practical. Cite supporting evidence using [E1], [E2], etc.'''.strip()

def generate_answer(query, contexts):
    contexts = [str(x).strip()[:MAX_CONTEXT_CHARS] for x in contexts[:TOP_K]]
    evidence = '\n\n'.join(f'[E{i+1}] {x}' for i,x in enumerate(contexts))
    messages = [
        {'role':'system','content':SYSTEM_PROMPT},
        {'role':'user','content':f'Farmer Question:\n{query}\n\nAgricultural Evidence:\n{evidence}\n\nProvide an evidence-grounded answer.'}
    ]
    inp = tokenizer.apply_chat_template(messages,tokenize=True,add_generation_prompt=True,return_tensors='pt',return_dict=True)
    inp = {k:v.to(llm.device) for k,v in inp.items()}
    if inp['input_ids'].shape[1] > MAX_INPUT_TOKENS:
        inp['input_ids'] = inp['input_ids'][:,-MAX_INPUT_TOKENS:]
        if 'attention_mask' in inp: inp['attention_mask'] = inp['attention_mask'][:,-MAX_INPUT_TOKENS:]
    n = inp['input_ids'].shape[1]; start = time.perf_counter()
    with torch.inference_mode():
        out = llm.generate(**inp,max_new_tokens=MAX_NEW_TOKENS,do_sample=False,use_cache=True,pad_token_id=tokenizer.eos_token_id,eos_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0,n:],skip_special_tokens=True).strip(), time.perf_counter()-start, n

GEN_PATH = SAVE_DIR/'G1_G2_Qwen_RAG_Full_Test.csv'
queries = test_df['query'].drop_duplicates().tolist()
if MAX_GENERATION_QUERIES is not None: queries = queries[:MAX_GENERATION_QUERIES]
if GEN_PATH.exists():
    previous = pd.read_csv(GEN_PATH); completed = set(previous['query'].dropna().astype(str)); generation_rows = previous.to_dict('records'); print('Resuming',len(completed),'completed queries.')
else:
    completed = set(); generation_rows = []
remaining = [q for q in queries if q not in completed]
print('Target generation queries:',len(queries),'| Remaining:',len(remaining))
for q in tqdm(remaining,desc='Qwen Original vs Confidence-Gated RAG'):
    g = test_df[test_df['query']==q].copy()
    base = g.sort_values('chunk_number').head(TOP_K); base_ctx = base['chunk'].tolist(); base_ans,base_lat,base_tok = generate_answer(q,base_ctx)
    gg = g[g['evidence_confidence']>=FINAL_GATE_THRESHOLD].sort_values('evidence_confidence',ascending=False).head(TOP_K); gated_ctx = gg['chunk'].tolist()
    if len(gated_ctx)==0:
        gated_ans='ABSTAIN: No evidence passed the confidence threshold.'; gated_lat=0.0; gated_tok=0; abst=1
    else:
        gated_ans,gated_lat,gated_tok = generate_answer(q,gated_ctx); abst=0
    resp = g['original_response'].dropna().astype(str).iloc[0] if ('original_response' in g.columns and g['original_response'].notna().any()) else None
    generation_rows.append({
        'query':q,'original_context_count':len(base_ctx),'gated_context_count':len(gated_ctx),
        'original_rag_answer':base_ans,'gated_rag_answer':gated_ans,
        'original_latency_s':base_lat,'gated_latency_s':gated_lat,
        'original_input_tokens':base_tok,'gated_input_tokens':gated_tok,
        'gate_abstained':abst,'dataset_response':resp,
        'original_context':'\n\n'.join(base_ctx),'gated_context':'\n\n'.join(gated_ctx)
    })
    pd.DataFrame(generation_rows).to_csv(GEN_PATH,index=False)
generation_df = pd.DataFrame(generation_rows)

# ---------------- FINAL PROXY EVALUATION ----------------
def split_sentences(t): return [x.strip() for x in re.split(r'(?<=[.!?])\s+',str(t).strip()) if x.strip()]
def citation_present(t): return int(bool(re.search(r'\[E\d+\]',str(t))))
def support_proxy(answer,context):
    ss=split_sentences(answer); cc=[x.strip() for x in str(context).split('\n\n') if x.strip()]
    if not ss or not cc: return np.nan
    a=bi_encoder.encode(ss,normalize_embeddings=True); c=bi_encoder.encode(cc,normalize_embeddings=True)
    return float((a@c.T).max(axis=1).mean())
def text_sim(a,b):
    if pd.isna(a) or pd.isna(b): return np.nan
    a=str(a).strip(); b=str(b).strip()
    if not a or not b: return np.nan
    e=bi_encoder.encode([a,b],normalize_embeddings=True); return float(np.dot(e[0],e[1]))

ge = generation_df.copy(); ge['original_citation_present']=ge['original_rag_answer'].apply(citation_present); ge['gated_citation_present']=ge['gated_rag_answer'].apply(citation_present)
osup=[]; gsup=[]; osim=[]; gsim=[]
for _,r in tqdm(ge.iterrows(),total=len(ge),desc='Evaluating generated answers'):
    osup.append(support_proxy(r['original_rag_answer'],r['original_context']))
    gsup.append(np.nan if int(r['gate_abstained']) else support_proxy(r['gated_rag_answer'],r['gated_context']))
    osim.append(text_sim(r['original_rag_answer'],r['dataset_response']))
    gsim.append(np.nan if int(r['gate_abstained']) else text_sim(r['gated_rag_answer'],r['dataset_response']))
ge['original_support_proxy']=osup; ge['gated_support_proxy']=gsup; ge['original_dataset_response_similarity']=osim; ge['gated_dataset_response_similarity']=gsim
end_summary = pd.DataFrame({
    'Metric':['Mean Context Count','Mean Latency (s)','Citation Presence Rate','Semantic Evidence-Support Proxy','Dataset-Response Similarity Proxy','Abstention Rate'],
    'G1 Original RAG':[ge['original_context_count'].mean(),ge['original_latency_s'].mean(),ge['original_citation_present'].mean(),ge['original_support_proxy'].mean(skipna=True),ge['original_dataset_response_similarity'].mean(skipna=True),0.0],
    'G2 Confidence-Gated RAG':[ge['gated_context_count'].mean(),ge['gated_latency_s'].mean(),ge['gated_citation_present'].mean(),ge['gated_support_proxy'].mean(skipna=True),ge['gated_dataset_response_similarity'].mean(skipna=True),ge['gate_abstained'].mean()]
})
display(end_summary.round(4))
ge.to_csv(SAVE_DIR/'G1_G2_Qwen_RAG_Evaluated.csv',index=False); end_summary.to_csv(SAVE_DIR/'G1_G2_End_to_End_Summary.csv',index=False)
summary = '\n'.join([
    'CAPSTONE CONFERENCE PAPER — AUTOMATIC RESULT SUMMARY','='*70,
    f'Dataset: {DATASET_REPO}',f'Judged query-chunk pairs: {len(eval_df)}',
    f'Train/Validation/Test queries: {train_df["query"].nunique()} / {val_df["query"].nunique()} / {test_df["query"].nunique()}',
    '', 'R0/R1/R2:', retrieval_summary.round(4).to_string(),
    '', f'Final confidence threshold: {FINAL_GATE_THRESHOLD:.2f}',
    '', 'Evidence Gate Test:', gate_summary.round(4).to_string(index=False),
    '', 'End-to-End Qwen:', end_summary.round(4).to_string(index=False),
    '', 'NOTE: Dataset-response similarity is a proxy, not gold-answer accuracy.',
    'NOTE: Semantic evidence-support is an embedding-based proxy, not a true faithfulness metric.'
])
(SAVE_DIR/'Conference_Paper_Automatic_Result_Summary.txt').write_text(summary,encoding='utf-8')
print(summary)
print('\nALL DONE. Results saved in:', SAVE_DIR)
